# Media Data Collection - Session 2: Extracting Information & Final Project

Welcome to the second session on Media Data Collection!

**Recap:** In Session 1, we learned how to fetch content from the web using the `requests` library, retrieving raw HTML from a webpage getting structured JSON data from an API. We also discussed the importance of ethical scraping.

**Goal:** Learn how to extract specific pieces of information from the raw HTML content we fetched, handle different extraction methods, understand the basics of dynamic websites, and walk through our final project crawler.

In this session, we will cover:
 1.  The need for parsing HTML.
 2.  Extracting data using:
     * Regular Expressions (Regex) - briefly.
     * HTML Parsing libraries: **BeautifulSoup** and **lxml/XPath**.
 3.  Understanding Dynamic Websites (conceptual).
 4.  **Final Project:** Walking through the Yahoo News Japan comment crawler code.

## 1. Recap & Introduction to Extraction

We know how to get the entire HTML source code of a page using `requests.get(url).text`. However, this gives us a large block of text containing HTML tags, CSS styles, JavaScript code, and the actual content we want (like headlines, article text, dates, names).

We need ways to **parse** this raw HTML and **extract** only the specific pieces of information relevant to our journalistic task. Imagine finding a specific quote within a book – you wouldn't just stare at the whole book; you'd use the chapter titles, index, or scan the pages for keywords. Parsing libraries help us do this programmatically.


## 2. Information Extraction Techniques

We'll look at three main techniques: Regex, BeautifulSoup, and lxml/XPath

### Installing Necessary Libraries

For this session and the final project, we need several libraries. Let's install them all now.
* `beautifulsoup4`: The main library for parsing HTML.
* `lxml`: A fast and robust HTML/XML parser that BeautifulSoup often uses underneath. Also provides XPath support.
* `requests`: We still need this for fetching.
* `pandas`: Excellent for organizing extracted data into tables (DataFrames).
* `retry`: A utility used in the final project code to automatically retry failed network requests.
* `tqdm`: Creates progress bars, useful for long-running scrapes (used in the final project).

In [ ]:
!pip install retry -q #tqdm beautifulsoup4 lxml requests pandas
print("Required libraries installed (or already exist).")

### Method 1: Regular Expressions (Regex) - Briefly

Regular Expressions are powerful tools for finding patterns in text. They define a search pattern using special characters. Python's built-in `re` module handles Regex.

**Example:** Find email addresses in a block of text.

In [ ]:
import re

text_block = """
Contact us at info@example.com for general inquiries.
For support, email support-team@example.org.
Or reach out to john.doe123@company.co.uk.
Not an email: info @ example . com
"""

# A basic regex pattern for finding email-like strings
# \S+ matches one or more non-whitespace characters
# @   matches the literal "@" symbol
# \.  matches a literal "." (dot needs escaping)
email_pattern = r'\S+@\S+\.\S+'

emails_found = re.findall(email_pattern, text_block)

print(f"Emails found using Regex: {emails_found}")

Emails found using Regex: ['info@example.com', 'support-team@example.org.', 'john.doe123@company.co.uk.']


**Pros:** Useful for finding well-defined patterns (emails, dates, phone numbers) in unstructured text.
**Cons:** **Generally NOT recommended for parsing HTML.** HTML has a nested structure, and slight changes in tags or attributes can easily break a Regex pattern. It quickly becomes complex and unreliable for extracting data based on HTML structure.

We might use Regex for specific tasks *within* scraped text (like cleaning up data or finding patterns in the content itself), or sometimes for extracting simple things from the raw HTML string if parsing fails (like the `userToken` in our final project), but parsing libraries are usually the right tool for navigating HTML.


### Method 2: HTML Parsing Libraries (BeautifulSoup & lxml/XPath)

 These libraries understand HTML structure. They convert the raw HTML text into a **tree** of objects representing tags, attributes, and content. This allows us to navigate the structure reliably.

 **HTML Tree (DOM - Document Object Model):** Think of an HTML document like a family tree.
 * The `<html>` tag is the root ancestor.
 * It has children like `<head>` and `<body>`.
 * `<body>` might have children like `<h1>` (heading), `<p>` (paragraph), `<div>` (container).
 * Tags can have **attributes** (like `class="headline"` or `id="main-content"` or `href="link_url"`).

 Parsing libraries let us say "Find all `<p>` tags inside the `<div>` with `id='main-content'`" instead of trying to match text patterns.

#### Using `BeautifulSoup`

`BeautifulSoup` is very popular due to its Pythonic feel and flexibility. It usually works best with a parser like `lxml`.


In [ ]:
import requests
from bs4 import BeautifulSoup

# Let's reuse the example URL from Session 1
url = "http://books.toscrape.com/"
headers = { # Remember headers!
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status() # Raise an error for bad status codes (4xx or 5xx)

    # Create a BeautifulSoup object
    # We pass the HTML content (response.text) and specify the parser ('lxml')
    soup = BeautifulSoup(response.text, 'lxml')
    print("HTML parsed successfully with BeautifulSoup!")

    # --- Example 1: Find the page title ---
    # The title is usually inside the <title> tag in the <head>
    page_title = soup.find('title').text # .text gets the content inside the tag
    print(f"\nPage Title: {page_title}")

    # --- Example 2: Find all book titles (which are links inside <h3> tags) ---
    # Inspecting the site shows book titles are in <a> tags within <h3> tags
    book_titles = []
    # find_all gets a list of all matching tags
    h3_tags = soup.find_all('h3')
    for h3 in h3_tags:
        link_tag = h3.find('a') # Find the <a> tag inside the <h3>
        if link_tag and link_tag.has_attr('title'): # Check if it exists and has a 'title' attribute
            book_titles.append(link_tag['title']) # Access attribute like a dictionary

    print(f"\nFound {len(book_titles)} book titles on the page:")
    for title in book_titles[:5]: # Print first 5
        print(f"  - {title}")

    # --- Example 3: Using CSS Selectors ---
    # BeautifulSoup also supports CSS selectors via .select() which can be very convenient
    # Find all price elements (they have class="price_color")
    price_elements = soup.select('p.price_color') # CSS selector for <p> tags with class "price_color"
    prices = [p.text for p in price_elements]
    print(f"\nFound {len(prices)} prices:")
    for price in prices[:5]: # Print first 5
        print(f"  - {price}")

except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")
except Exception as e:
    print(f"An error occurred during parsing: {e}")

HTML parsed successfully with BeautifulSoup!

Page Title: 
    All products | Books to Scrape - Sandbox


Found 20 book titles on the page:
  - A Light in the Attic
  - Tipping the Velvet
  - Soumission
  - Sharp Objects
  - Sapiens: A Brief History of Humankind

Found 20 prices:
  - Â£51.77
  - Â£53.74
  - Â£50.10
  - Â£47.82
  - Â£54.23


**Key `BeautifulSoup` Methods:**
* `BeautifulSoup(html_content, 'parser_name')`: Creates the soup object. Common parsers: `'lxml'`, `'html.parser'` (built-in).
* `soup.find('tag_name', attribute='value')`: Finds the *first* matching tag.
* `soup.find_all('tag_name', class_='css_class_name')`: Finds *all* matching tags. (Note: `class_` has an underscore because `class` is a Python keyword).
* `soup.select('css_selector')`: Finds all tags matching a CSS selector (e.g., `div#main p.content`, `a[href]`). Very powerful.
* `.text`: Gets the text content inside a tag (and its children).
* `tag['attribute_name']`: Gets the value of an attribute (e.g., `link_tag['href']`).
* `tag.has_attr('attribute_name')`: Checks if a tag has a specific attribute.

#### Brief Mention: `lxml` and XPath

`lxml` is the underlying parser we used with BeautifulSoup. It can also be used directly and supports **XPath**, another language (besides CSS selectors) for navigating the HTML/XML tree. XPath is very powerful, especially for complex selections.

The final project code uses `lxml` and XPath extensively.

 **XPath Syntax Basics:**
 * `//tagname`: Selects all tags with that name anywhere in the document.
 * `/tagname`: Selects direct children with that name.
 * `[@attribute='value']`: Selects tags with a specific attribute value.
 * `/text()`: Selects the text content of a tag.
 * `.`: Selects the current node.
 * `..`: Selects the parent node.


In [ ]:
# Example using lxml and XPath directly (optional comparison)
from lxml import html

try:
    # Re-fetch or reuse response.text from the previous cell
    if 'response' in locals() and response and response.status_code == 200:
        # Parse the HTML using lxml
        tree = html.fromstring(response.text)
        print("\nHTML parsed successfully with lxml!")

        # --- Example 1 using XPath: Find page title ---
        # XPath selects the text() directly inside the //head/title path
        xpath_title = tree.xpath('//head/title/text()')[0] # XPath often returns a list
        print(f"Page Title (XPath): {xpath_title}")

        # --- Example 2 using XPath: Find all book titles ---
        # Select the 'title' attribute of <a> tags within <h3> tags
        xpath_book_titles = tree.xpath('//h3/a/@title') # @ gets the attribute value
        print(f"\nFound {len(xpath_book_titles)} book titles (XPath):")
        for title in xpath_book_titles[:5]:
             print(f"  - {title}")

        # --- Example 3 using XPath: Find all prices ---
        # Select text of <p> tags with class 'price_color'
        xpath_prices = tree.xpath('//p[@class="price_color"]/text()')
        print(f"\nFound {len(xpath_prices)} prices (XPath):")
        for price in xpath_prices[:5]:
            print(f"  - {price}")

    else:
        print("\nCould not run lxml example, previous request failed.")

except Exception as e:
    print(f"\nAn error occurred during lxml/XPath parsing: {e}")



HTML parsed successfully with lxml!
Page Title (XPath): 
    All products | Books to Scrape - Sandbox


Found 20 book titles (XPath):
  - A Light in the Attic
  - Tipping the Velvet
  - Soumission
  - Sharp Objects
  - Sapiens: A Brief History of Humankind

Found 20 prices (XPath):
  - Â£51.77
  - Â£53.74
  - Â£50.10
  - Â£47.82
  - Â£54.23


In [ ]:
title_path = '//*[@id="default"]/div/div/div/div/section/div[2]/ol/li/article/h3/a'
titles_elements = tree.xpath(title_path)
title_texts = [ele.text for ele in titles_elements]

Both `BeautifulSoup` (with CSS Selectors or `find`/`find_all`) and `lxml` (with XPath) are excellent tools for parsing HTML. The choice often depends on personal preference or the specific complexity of the selectors needed. Our final project code primarily uses `lxml` and XPath.

## 3. Handling Dynamic Websites (Conceptual)

 Sometimes, when you view the source (`response.text`) of a page using `requests`, you'll notice that the content you see in your browser isn't actually there!

 This often happens with **Dynamic Websites**. These sites load a basic HTML structure initially, and then use **JavaScript** code running in your browser to fetch data (often from APIs) and update the page *after* the initial load.

 **Why `requests` Fails:** `requests` only fetches the initial HTML source code sent by the server. It does *not* run JavaScript. Therefore, it won't see the content loaded dynamically.

 **How to Handle Dynamic Content:**

 1.  **Check the Browser's Network Tab:**
     * Open your browser's Developer Tools (usually F12 or right-click -> Inspect).
     * Go to the "Network" tab.
     * Reload the page.
     * Look for requests (often `XHR` or `Workspace`) that seem to be loading the data you want. You might find a hidden API endpoint!
     * If you find an API, you can try calling it directly using `requests` (like we did with JSONPlaceholder). This is often the most efficient method if available. **Our final project uses this approach to get comment replies.**
 2.  **Use Browser Automation Tools (e.g., Selenium, Playwright):**
     * These tools control an actual web browser (like Chrome or Firefox) programmatically.
     * They load the page, wait for JavaScript to run, and then let you extract the content from the fully rendered page.
     * **Pros:** Can handle almost any dynamic website.
     * **Cons:** Slower, more resource-intensive (runs a full browser), can be more complex to set up.
     * *We won't cover Selenium in detail in this course, but it's the next step for complex JavaScript-heavy sites where you can't find an underlying API.*

## 4. Final Project: Yahoo News Japan Comment Crawler

 Now, let's put everything together and look at the crawler designed to fetch comments from a Yahoo News Japan article page.

**Goal:** Given the URL of a specific Yahoo News Japan article, scrape all the comments and their replies, including details like the commenter, timestamp, comment text, and vote counts.

**Approach Used in the Code:**
 1.  **Fetch Initial Page:** Use `requests` and `lxml` to fetch the comments page (`article_url/comments`).
 2.  **Parse HTML:** Use `lxml` and **XPath** to extract the comments visible on the first page.
 3.  **Extract `userToken`:** Use **Regex** (`re.findall`) on the raw HTML to find a special `userToken` needed to make API calls for replies. (This is an example where Regex is used carefully on HTML source).
 4.  **Handle Replies (API):** For comments showing replies, use the extracted `userToken` and a `requests.Session` to call a **hidden API endpoint** (likely found using the Network Tab method) that returns reply data in **JSON** format. Parse the JSON.
 5.  **Handle Pagination:** Find the 'Next' button using **BeautifulSoup** (`.find_all` with CSS class) on the comments page. Loop through subsequent pages, repeating steps 1-4 for each page.
 6.  **Structure Data:** Store all collected comment and reply data in lists and then combine them into a **Pandas DataFrame** for easy analysis and saving.
 7.  **Helper Functions:** Use functions for common tasks like making requests (`get_session_response` with retry logic) and processing timestamps (`process_timestamp`).

 ### Code Integration & Walkthrough

 Below is the Python code for a comment crawler of [Yahoo! News Japan](https://news.yahoo.co.jp/). Read the comments carefully to understand each part.

In [ ]:
# Note: Libraries were installed in a previous cell.
!pip install retry -q

import requests
from bs4 import BeautifulSoup # Used for finding pagination button
import re                 # Used for extracting userToken
from lxml import html      # Used for primary HTML parsing via XPath
from retry import retry    # Used for retrying failed requests
from tqdm.notebook import tqdm # For progress bars (optional but nice)
import time               # For adding delays between requests
from datetime import datetime, timedelta, timezone # For handling timestamps
import pandas as pd       # For organizing data into tables
import os                 # Potentially used for saving files (commented out)
import json               # For handling JSON data from API

print("Libraries imported.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 kB 2.4 MB/s eta 0:00:00
Libraries imported.


In [ ]:
# Define headers to make requests look like they come from a standard browser.
# This can help avoid being blocked by some websites.

headers = {
    'User-Agent': "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/104.0.0.0 Safari/537.36", # Example User Agent
    "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9",
    # Other headers like accept-language could also be added if needed.
}
print("Headers defined.")

Headers defined.


In [ ]:
# --- Function to get response using requests.Session and lxml parsing ---
# Includes retry logic using the @retry decorator
# Creates a Session object, which can persist cookies and improve performance for multiple requests.
@retry(tries=3, delay=5, backoff=2) # Retry 3 times, wait 5s, then 10s, then 20s
def get_session_response(url, headers, session=None, delay=0.5):
    """
    Fetches content from a URL using a requests.Session, parses with lxml,
    and includes retry logic.
    Returns the parsed lxml tree, the response object, and the session object.
    """
    if not session:
        session = requests.Session() # Create a new session if one isn't provided
        session.headers.update(headers) # Update session headers

    print(f"Fetching: {url}...")
    time.sleep(delay) # Polite delay before making request
    response = session.get(url, timeout=20) # Increased timeout for potentially slow pages
    response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)

    # Decode explicitly using UTF-8, common for Japanese sites
    response.encoding = 'utf-8'
    text_content = response.text

    # Parse the HTML content using lxml
    tree = html.fromstring(text_content)
    print(f"Successfully fetched and parsed.")
    return tree, response, session

In [ ]:
# --- Function to process Yahoo News timestamps ---
def process_timestamp(timestamp_str, current_date):
    """
    Converts Yahoo News comment timestamps (like '5分前', '1/3(水) 15:01', 'たった今')
    into a standard 'YYYY-MM-DD HH:MM:SS' format.
    """
    JST = timezone(timedelta(hours=+9), 'JST') # Japan Standard Time

    if not isinstance(timestamp_str, str): # Handle potential non-string input
        return None

    timestamp_str = timestamp_str.strip() # Remove leading/trailing whitespace

    try:
        if timestamp_str == 'たった今': # "Just now"
            date = current_date
            return date.strftime('%Y-%m-%d %H:%M:%S')

        # Check for relative timestamps (e.g., '5日前', '8時間前', '10分前', '5秒前')
        relative_match = re.search(r'(\d+)(日前|時間前|分前|秒前)', timestamp_str)
        if relative_match:
            time_quantity, time_unit = int(relative_match.group(1)), relative_match.group(2)
            if time_unit == '日前':
                date = current_date - timedelta(days=time_quantity)
            elif time_unit == '時間前':
                date = current_date - timedelta(hours=time_quantity)
            elif time_unit == '分前':
                date = current_date - timedelta(minutes=time_quantity)
            elif time_unit == '秒前':
                date = current_date - timedelta(seconds=time_quantity)
            else: # Should not happen with the regex
                 return timestamp_str # Return original if unit unknown
            return date.strftime('%Y-%m-%d %H:%M:%S')

        # Check for absolute timestamps (e.g., '1/3(水) 15:01')
        # This pattern assumes the format Month/Day(DayOfWeek) Hour:Minute
        absolute_match = re.search(r'(\d{1,2})/(\d{1,2})\(.\)\s+(\d{1,2}):(\d{1,2})', timestamp_str)
        if absolute_match:
            month, day, hour, minute = map(int, absolute_match.groups())
            # Handle year transition: if the comment month is later than the current month, assume last year.
            year = current_date.year
            if current_date.month < month:
                year -= 1
            # Create datetime object, assuming JST
            date = datetime(year, month, day, hour, minute, tzinfo=JST)
            return date.strftime('%Y-%m-%d %H:%M:%S')

    except Exception as e:
        print(f"Error processing timestamp '{timestamp_str}': {e}")
        return timestamp_str # Return original string if parsing fails

    # If no pattern matched, return the original string
    return timestamp_str

In [ ]:
def reply_collector(cid, rn, userToken, session, news_id, progressbar = False, delay = 0.5):
  replies_data = [pd.DataFrame([])]
  JST = timezone(timedelta(hours=+9), 'JST')

  start = 1
  api_url = "https://news.yahoo.co.jp/api/comment/properties/news_user/articles/{news_id}/comments/{cid}/reply?start={start}&results=10&sort=recommendation"
  session.headers.update({"Usertoken":userToken})
  response = session.get(api_url.format(news_id=news_id,cid = cid, start = start))
  time.sleep(delay)
  current_date = datetime.now(JST)

  if progressbar: pb = tqdm()
  if response.status_code == 200:
    replies = response.json()
    replies_df = pd.DataFrame(replies.get("comments",[]))
    #print(replies)
    #print(response.text)

    replies_df["timestamp"] = replies_df.postDate.apply(process_timestamp, args=(current_date,))
    replies_df = replies_df[["commentId","timestamp","text","name","ppid","empathyCount","insightCount","negativeCount", "replyTo"]]
    replies_df.columns = ["comment_id","timestamp","comment","publisher","publisher_id","sympathizing_num","maker_sense_num","doubt_num", "replyTo"]
    replies_df["news_id"] = news_id
    replies_df["current_date"] = datetime.strftime(current_date,'%Y-%m-%d')
    replies_df["reply_num"] = "0"

    replies_data.append(replies_df)

    if progressbar: pb.update(len(replies_df))
    if progressbar: pb.set_description(f"collecting replies from comment: {cid}")

    totalResults = replies.get("totalResults",0)

    while totalResults > (start+10):
      start += 10
      response = session.get(api_url.format(news_id=news_id,cid = cid, start = start))
      time.sleep(delay)
      current_date = datetime.now(JST)
      if not response.status_code == 200:
        break

      replies = response.json()
      replies_df = pd.DataFrame(replies.get("comments",[]))
      replies_df["timestamp"] = replies_df.postDate.apply(process_timestamp, args=(current_date,))
      replies_df = replies_df[["commentId","timestamp","text","name","ppid","empathyCount","insightCount","negativeCount", "replyTo"]]
      replies_df.columns = ["comment_id","timestamp","comment","publisher","publisher_id","sympathizing_num","maker_sense_num","doubt_num", "replyTo"]
      replies_df["news_id"] = news_id
      replies_df["current_date"] = datetime.strftime(current_date,'%Y-%m-%d')
      replies_df["reply_num"] = "0"

      replies_data.append(replies_df)

      if progressbar: pb.update(len(replies_df))
      if progressbar: pb.set_description(f"collecting replies from comment: {cid}")


  return pd.concat(replies_data)

def expert_data_extractor(soup, news_id, current_date):
  expert_data = []
  for expert_info in soup.find_all("div", class_ = "sc-z8tf0-2 eRySMu"):
    expert_name = "UNKNOWN"
    expert_url = "UNKNOWN"
    timestamp = "UNKNOWN"
    affiliation = "UNKNOWN"
    comment_type = "UNKNOWN"
    support_num = "UNKNOWN"
    expert_identity = expert_info.find("h2", "sc-z8tf0-4 ftkuJs")
    if expert_identity:
      expert_name = expert_identity.text
      expert_url =  expert_identity.find("a").attrs["href"]
    ts = expert_info.find("time", class_="sc-z8tf0-7 llqEPu")
    if ts:
      timestamp = process_timestamp(ts.text,current_date)
    affiliation_info = expert_info.find("p", class_="sc-z8tf0-9 kHCbXc")
    if affiliation_info:
      affiliation = affiliation_info.text

    comment_type_info = expert_info.find("span")
    if comment_type_info:
      comment_type = comment_type_info.text
    support_num_info = expert_info.find("span", class_ = "sc-x1bzsn-8 caCESi")
    if support_num_info:
      support_num = support_num_info.text

    expert_data.append({"expert_name": expert_name, "expert_url": expert_url, "timestamp": timestamp, "affiliation": affiliation, "comment_type": comment_type, "support_num": support_num})
  expert_df = pd.DataFrame(expert_data)
  if not expert_df.empty:
    expert_df["news_id"] = news_id
  else:
    expert_df = pd.DataFrame([])
  return expert_df
  #expert.find("div")

In [ ]:
# This function orchestrates the process for a given news article URL.
def comments_crawler(nurl,path = "",content_save = False):
  comment_next_path = '//*[@id="comment-main"]/div[6]/div/ul/li/a'
  comment_content_path = '//*[@id="comment-main"]/ul[2]/li/article/article/div[2]/p'
  comment_date_path = '//*[@id="comment-main"]/ul[2]/li/article/article/div[2]/div[1]/time/a'
  comment_publisher_path = '//*[@id="comment-main"]/ul[2]/li/article/article/div[2]/div[1]/h2/a'
  comment_id_path = '//*[@id="comment-main"]/ul[2]/li/article/article/div[2]/div[1]/time/a'

  replies_path = '//*[@id="comment-main"]/ul[2]/li/article/article/div[2]/div[2]/button[2]/span'
  sympathizing_path = '//*[@id="comment-main"]/ul[2]/li/article/article/div[2]/div[2]/div/ul/li[1]/button[2]/span/span[2]'
  make_sense_path = '//*[@id="comment-main"]/ul[2]/li/article/article/div[2]/div[2]/div/ul/li[2]/button[2]/span/span[2]'
  doubt_path = '//*[@id="comment-main"]/ul[2]/li/article/article/div[2]/div[2]/div/ul/li[3]/button[2]/span/span[2]'

  page = 1
  news_id = re.search(r'articles/(.+)',nurl).group(1)
  JST = timezone(timedelta(hours=+9), 'JST')

  replies = []

  html_container =  {}
  html_container[news_id] = []

  comment_url =  nurl + '/comments' #'https://news.yahoo.co.jp/articles/040dfc71f4ef202a26d41daf2b3ffe213f8e2bbe/comments?page=5'
  tree, response, session = get_session_response(comment_url,headers)
  text = response.content.decode("utf-8")
  userToken = re.findall(r'"userToken":"(.*?)"', text, re.DOTALL)
  soup = BeautifulSoup(response.text)
  next_button = soup.find_all("a", class_ = "sc-brfqoi-5 fxZmTZ")
  #text_ = next_button[-1].text if next_button else "NONE"

  current_date = datetime.now(JST)
  expert_df = expert_data_extractor(soup, news_id, current_date)

  if userToken:
    userToken = userToken[0]

  current_date = datetime.now(JST)
  if content_save: html_container[news_id].append(response.text) #save_content_as_html(response.text, os.path.join(path, "comments_{}_{}.html".format(news_id,page)))

  timestamps = []
  comments = []
  publishers = []
  publishers_id = []
  current_dates = []
  comments_id = []
  reply_num = []
  sympathizing_num = []
  doubt_num = []
  make_sense_num = []

  coms = pd.DataFrame([])
  if tree.xpath(comment_date_path):

    cids = [com.attrib['href'].split("/")[-1] for com in tree.xpath(comment_id_path)]
    comments_id.extend(cids)

    timestamps.extend([process_timestamp(com.text,current_date) for com in tree.xpath(comment_date_path)])
    comments.extend([com.text for com in tree.xpath(comment_content_path)])
    publishers.extend([com.text for com in tree.xpath(comment_publisher_path)])
    publishers_id.extend([com.attrib['href'].split("/")[-1] for com in tree.xpath(comment_publisher_path)])
    current_dates.extend([datetime.strftime(current_date,'%Y-%m-%d') for com in tree.xpath(comment_date_path)])

    rns = [reply.text for reply in tree.xpath(replies_path)]
    reply_num.extend(rns)
    sympathizing_num.extend([sympath.text for sympath in tree.xpath(sympathizing_path)])
    make_sense_num.extend([make_sense.text for make_sense in tree.xpath(make_sense_path)])
    doubt_num.extend([doubt.text for doubt in tree.xpath(doubt_path)])

    replied_comments = [(cid, rn) for cid, rn in zip(cids, rns) if not rn == "0"]
    #print(f"userToken: {userToken}")
    #print(len(replied_comments))
    for cid, rn in replied_comments:
      try:
        #print(cid, rn)
        replies_data = reply_collector(cid, rn, userToken, session, news_id, progressbar = False)
        replies.append(replies_data)
      except Exception as err:
        print(f"failed when collecting replies of comment {cid}")
        print(err)

    page = 2
    #while next_button == '次へ':
    text_ = next_button[-1].text if next_button else "NONE"
    while text_ == '次へ':
      next_url =  next_button[-1].attrs["href"] # #
      print(next_url)
      nurl = "https://news.yahoo.co.jp" + next_url
      tree, response,session = get_session_response(nurl,headers,delay=0.5)
      text = response.content.decode("utf-8")
      soup = BeautifulSoup(response.text)
      userToken = re.findall(r'"userToken":"(.*?)"', text, re.DOTALL)
      if userToken:
        userToken = userToken[0]

      current_date = datetime.now(JST)

      cids = [com.attrib['href'].split("/")[-1] for com in tree.xpath(comment_id_path)]
      comments_id.extend(cids)

      timestamps.extend([process_timestamp(com.text,current_date) for com in tree.xpath(comment_date_path)])
      comments.extend([com.text for com in tree.xpath(comment_content_path)])
      publishers.extend([com.text for com in tree.xpath(comment_publisher_path)])
      publishers_id.extend([com.attrib['href'].split("/")[-1] for com in tree.xpath(comment_publisher_path)])
      current_dates.extend([datetime.strftime(current_date,'%Y-%m-%d') for com in tree.xpath(comment_date_path)])

      rns = [reply.text for reply in tree.xpath(replies_path)]
      reply_num.extend(rns)
      sympathizing_num.extend([sympath.text for sympath in tree.xpath(sympathizing_path)])
      make_sense_num.extend([make_sense.text for make_sense in tree.xpath(make_sense_path)])
      doubt_num.extend([doubt.text for doubt in tree.xpath(doubt_path)])

      replied_comments = [(cid, rn) for cid, rn in zip(cids, rns) if not rn == "0"]
      for cid, rn in replied_comments:
        try:
          replies_data = reply_collector(cid, rn, userToken, session, news_id, progressbar = False)
          replies.append(replies_data)
        except Exception as err:
          print(f"failed when collecting replies of comment {cid}")
          print(err)
          #slack_client.chat_postMessage(
          #  channel="#newscrawler",
          #  text=f"failed when collecting replies of comment {cid}")

          #slack_client.chat_postMessage(
          #  channel="#newscrawler",
          #  text=err)

      if content_save: html_container[news_id].append(response.text) #save_content_as_html(response.text, os.path.join(path, "comments_{}_{}.html".format(news_id,page)))

      next_button = soup.find_all("a", class_ = "sc-brfqoi-5 fxZmTZ")
      text_ = next_button[-1].text if next_button else "NONE"
      page +=1
    coms = pd.DataFrame({"comment_id": comments_id,'timestamp':timestamps,'comment':comments\
                         ,'publisher':publishers,"publisher_id":publishers_id,"current_date":current_dates\
                         ,"reply_num": reply_num,"sympathizing_num": sympathizing_num,"maker_sense_num": make_sense_num, "doubt_num": doubt_num})
    coms["news_id"] = news_id
    coms["replyTo"] = ""
  comment_data = pd.concat(replies+[coms])
  return comment_data, html_container, expert_df

In [ ]:
# --- !!! IMPORTANT !!! ---
# Replace this URL with a *current* Yahoo News Japan article URL that has comments.
# URLs from old articles might not work or might have no comments left.
# Find an article on https://news.yahoo.co.jp/ and copy its URL here.
# Example format: https://news.yahoo.co.jp/articles/xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx

#target_article_url = "https://news.yahoo.co.jp/articles/PASTE_A_VALID_YAHOO_NEWS_JAPAN_ARTICLE_URL_HERE"
target_article_url = "https://news.yahoo.co.jp/articles/6368069665d8c022d179335fed834f7242fe6840"
# --- Check if a placeholder URL is still present ---
if "PASTE_A_VALID_YAHOO_NEWS_JAPAN_ARTICLE_URL_HERE" in target_article_url:
    print("="*50)
    print("PLEASE REPLACE THE PLACEHOLDER URL in the 'target_article_url' variable")
    print("with a real, current Yahoo News Japan article URL that has comments.")
    print("You can find one on https://news.yahoo.co.jp/")
    print("="*50)
else:
    comment_data, html_container, expert_df = comments_crawler(target_article_url)

Fetching: https://news.yahoo.co.jp/articles/f4a64aad30bfbf167ee5758da908a47007390d19/comments...
Successfully fetched and parsed.
/articles/f4a64aad30bfbf167ee5758da908a47007390d19/comments?page=2
Fetching: https://news.yahoo.co.jp/articles/f4a64aad30bfbf167ee5758da908a47007390d19/comments?page=2...
Successfully fetched and parsed.
/articles/f4a64aad30bfbf167ee5758da908a47007390d19/comments?page=3
Fetching: https://news.yahoo.co.jp/articles/f4a64aad30bfbf167ee5758da908a47007390d19/comments?page=3...
Successfully fetched and parsed.
/articles/f4a64aad30bfbf167ee5758da908a47007390d19/comments?page=4
Fetching: https://news.yahoo.co.jp/articles/f4a64aad30bfbf167ee5758da908a47007390d19/comments?page=4...
Successfully fetched and parsed.


In [ ]:
comment_data

,comment_id,timestamp,comment,publisher,publisher_id,sympathizing_num,maker_sense_num,doubt_num,replyTo,news_id,current_date,reply_num
0,3e084ad5-f2f1-420e-a323-ad6b055d93b5,2025-05-05 21:29:04,私は昨日は明治村を歩いていて少し暑かったですが、今日は半袖では肌寒く感じています。\n\n確...,chu********,vsWV5l29VD1jm-LFInzN4tHQzZoysJZxny43QgHv71u-ix...,3,0,0,0fc5d802-2049-406c-86fc-0a9c61b73081,f4a64aad30bfbf167ee5758da908a47007390d19,2025-05-05,0
1,410daab0-5703-4814-946d-e67d585e8c10,2025-05-05 22:03:04,夏の酷暑が近づいてくると今年も無事に生き延びることができるんだろうかと不安がよぎる,iko********,AhXXhmBbxXqCUV1ASGSF9JQCQUk4Oif331zhCUF4PkwCW1...,0,0,0,0fc5d802-2049-406c-86fc-0a9c61b73081,f4a64aad30bfbf167ee5758da908a47007390d19,2025-05-05,0
2,f25258d9-4e75-4db0-883f-c9eb2344a1bd,2025-05-05 22:03:04,おもしろい文章ですね。\n小説を読んでるみたい。\n明日は12時からは晴れて欲しいです。\n...,qcd********,Krc9LfJtMsNk_yjZn7_zqW1cJDdn2D9vCGJlLo03dn4q-b...,1,0,0,0fc5d802-2049-406c-86fc-0a9c61b73081,f4a64aad30bfbf167ee5758da908a47007390d19,2025-05-05,0
3,c7ff647e-75a1-4c63-a0d6-c755ffa4cb12,2025-05-05 21:29:04,梅雨の時期は\n上着が必要になるよ,ky7********,B_jj1nW2xX4ivUaeD4g27HxUmOMpmLO40Z5WJ66msoIHtm...,6,2,2,0fc5d802-2049-406c-86fc-0a9c61b73081,f4a64aad30bfbf167ee5758da908a47007390d19,2025-05-05,0
0,ea4a96f2-9171-459f-b3ff-ccd3031a09b3,2025-05-05 21:24:05,今の所例年より全然涼しい日が続いてますもんね〜、、。\n\n流石に晴れてる日は昼過ぎ〜夕方ま...,asc********,IdIpU9cZcCnRO29eaWEnZCpeaaFPBASpcFMbNbUWANUhnK...,19,0,1,67f74e4c-7edf-47a2-a00b-5b73bcc3d79d,f4a64aad30bfbf167ee5758da908a47007390d19,2025-05-05,0
0,e1617bd9-53f2-4d17-bf30-b9313a71a1f4,2025-05-05 21:28:05,>>お昼寝さん\n\nそうですか。それで？\n\n止められます？,mep********,YkIH9eKJp39eC6i_0u8L5Inq1zN8p5pSAldv3uroUB1iDI...,4,0,4,d6269d32-e681-4346-9d3c-cf0bfb037fa8,f4a64aad30bfbf167ee5758da908a47007390d19,2025-05-05,0
1,f14594d2-bf06-4283-b98d-e5ba2c5d0a66,2025-05-05 21:21:05,ここは君のGWの出来事や感想文書く場所でない\n\nインスタでやってくれ,お昼寝,nZikLASk4Fyww_CLr8K8F5RM7pPG3yjrO9IKZOKHmESd1i...,4,0,19,d6269d32-e681-4346-9d3c-cf0bfb037fa8,f4a64aad30bfbf167ee5758da908a47007390d19,2025-05-05,0
0,36de2f28-278e-4fe4-a24d-ce53ae77b653,2025-05-05 21:35:06,半袖で十分な日がほとんどないですね。,削除されそうなコメントです,ijmmRxhN0Q4TMfmp5IH8q161aSo65xcK5yeE05AfJDCKdy...,6,0,1,7189fb2b-8e20-4b45-9515-b9e7c8e52b1b,f4a64aad30bfbf167ee5758da908a47007390d19,2025-05-05,0
0,7e98aba7-0b3a-4d7c-8bd3-4003579a4ef2,2025-05-05 21:33:07,優勝争いには程遠い成績なので若手育成のためチャンスをたくさん与えるのに好都合ですね～。,iun********,5Q4p22OdJ1PQSkINsPHHeCTVHce-m_8L7d-orUY93qXlQK...,2,1,0,ff119ae2-675d-41f3-ad7b-34a26f481717,f4a64aad30bfbf167ee5758da908a47007390d19,2025-05-05,0
0,364a8302-2e5c-44ec-9f02-3abaf2786127,2025-05-05 21:37:09,ほのぼの(*´∀｀*),os1********,TNF7kWs0OPoMaqHGtaMBRs8dU41-sfsN2GudfFIwAu9Mn_...,10,0,0,5b9d8c43-b0c7-4ab5-97d6-7463fa3d6803,f4a64aad30bfbf167ee5758da908a47007390d19,2025-05-05,0


### Discussion & Potential Issues
  * **Website Changes:** This crawler relies heavily on specific XPath selectors and CSS classes (`sc-brfqoi-5 fxZmTZ`, etc.) and the format of the reply API. If Yahoo News Japan changes its website structure or API, **this crawler will likely break**. Web scrapers often require maintenance.
  * **Getting Blocked:** Sending too many requests too quickly, or scraping aggressively, can lead to your IP address being temporarily or permanently blocked. The delays (`time.sleep()`) and retry logic help mitigate this, but are not foolproof. Using proxies or more sophisticated scraping frameworks might be necessary for large-scale scraping.
  * **API Rate Limits:** The hidden reply API might have unpublished rate limits. Exceeding them could lead to errors or blocks.
  * **JavaScript Challenges:** If Yahoo changes comments to load *entirely* via JavaScript without a discoverable API, this `requests`-based approach would fail, and tools like Selenium would be needed.
  * **Ethical Use:** Remember the ethical guidelines. Use this for specific data collection tasks relevant to reporting, not for overwhelming the site or misusing the data.

## 5. Conclusion & Further Learning

 Congratulations! You've now seen the entire process from fetching raw web content to parsing it and extracting structured data, even handling some dynamic aspects via APIs.

 **Key Skills Recap:**
 * Fetching web content (`requests`).
 * Understanding and using APIs (fetching JSON).
 * Parsing HTML (`BeautifulSoup`, `lxml`).
 * Navigating HTML structure (CSS Selectors, XPath).
 * Extracting specific data elements.
 * Handling pagination (finding 'Next' links).
 * Basic use of Regex for specific extraction (`userToken`).
 * Structuring collected data (`pandas`).

 **Where to Go Next?**
 * **Practice:** Try modifying the final project code to extract different information, or try scraping comments from another (simpler) website.
 * **Library Documentation:** Explore the docs for `requests`, `BeautifulSoup`, `lxml`, and `pandas` to learn more advanced features.
 * **Selenium/Playwright:** If you need to scrape heavily JavaScript-dependent sites, investigate browser automation tools.
 * **Scrapy Framework:** For building larger, more complex, and efficient crawlers, look into the Scrapy framework for Python.
 * **Web Developer Tools:** Get proficient with your browser's Developer Tools (Inspect Element, Network Tab) – they are essential for understanding how websites work and planning your scraping strategy.

 This course provided a foundation. Web scraping is a deep field, but you now have the basic knowledge and tools to start collecting valuable data from the web for your journalistic work. Good luck!